# Global Weather Forecasts and Wind Particles with geeViz

Forecast weather from three global models, with an animated wind field
in the style of windy.com.

| Model | `model` | Collection | Resolution | Reaches forward |
|---|---|---|---|---|
| NOAA GFS | `gfs` | `NOAA/GFS0P25` | 0.25° | ~16 days |
| ECMWF IFS | `euro` | `ECMWF/NRT_FORECAST/IFS/OPER` | ~0.4° | ~6 days |
| WeatherNext 3 | `weathernext` | `.../weathernext_3_0_0_0p1deg` | 0.1° | ~15 days |

`geeViz.weather` puts all three behind one call. Verified against the
live collections on 2026-09-10.

Each model marks time differently — GFS and ECMWF use
`creation_time` / `forecast_time` as epoch milliseconds, WeatherNext
uses `start_time` / `end_time` as ISO 8601 strings. Reconciling that is
what `getForecastData` is for, so you can ask for a date range and get
the right images regardless of model.

Copyright 2026 Ian Housman

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

   http://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.

[![github](https://img.shields.io/badge/-see%20sources-white?logo=github&labelColor=555)](https://github.com/gee-community/geeviz/blob/master/examples/weather_forecast_examples.ipynb)
[![github](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gee-community/geeViz/blob/master/examples/weather_forecast_examples.ipynb)

## Setup

In [1]:
import datetime

try:
    import geeViz.geeView as gv
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'geeViz'])
    import geeViz.geeView as gv

import geeViz.weather as wx

ee = gv.ee
Map = gv.Map
Map.clearMap()
Map.setQueryDateFormat('YYYY-MM-dd HH:mm')

now = datetime.datetime.now(datetime.timezone.utc)
fut_start = (now + datetime.timedelta(days=1)).strftime('%Y-%m-%d')
fut_end   = (now + datetime.timedelta(days=3)).strftime('%Y-%m-%d')
# Hurricane Helene (2024) - made landfall: September 23, 2024
past_start = '2024-09-26'
past_end   = '2024-09-27'

study_area = ee.Geometry.Rectangle([-125, 31, -102, 49])   # US West
Map.centerObject(study_area, 5)

# WeatherNext is a gated dataset. Check access up front so an unentitled
# account gets a request-access URL rather than a cryptic error deep in
# the pipeline. GFS and ECMWF are open and always run.
HAS_WN = False
try:
    ee.ImageCollection(wx.MODELS['weathernext']['collection']).limit(1).size().getInfo()
    HAS_WN = True
except Exception as e:
    print(f'[preflight] WeatherNext not readable: {type(e).__name__}')
    print('           Request access: https://developers.google.com/weathernext/guides/earth-engine')

print(f'forward {fut_start}..{fut_end}   past {past_start}..{past_end}   WeatherNext={HAS_WN}')

[geeViz.eeAuth] EE initialized via proxy: http://127.0.0.1:8889/ee-api (attached to existing detached proxy, tenant_header=X-geeViz-Creds)


geeViz: Earth Engine ready (project='geeviz-geo-agent', source=eeauth-proxy)
Setting default query date format to: YYYY-MM-dd HH:mm
forward 2026-09-11..2026-09-13   past 2024-09-26..2024-09-27   WeatherNext=True


## 1. Picking the right images

`getForecastData(startDate, endDate, model)` chooses based on where the
window sits relative to now:

| window | what you get |
|---|---|
| **past** | the shortest-lead image from every run initialized in the window — one fresh analysis per cycle, the best record of what actually happened |
| **future** | a single run: the most recent one that REACHES the end of your window, filtered to it |
| **spanning now** | analyses up to the most recent initialization, then that run's forecast from there — merged |

"Shortest lead" rather than literally lead 0: GFS and ECMWF publish a
0-hour analysis, WeatherNext's `forecast_hour` runs 1..360 and never
reaches 0. The value is read from each collection rather than assumed.

The lead hours below are the proof.

In [2]:
# Every frame's system:time_start, so you can see it is the VALID time.
# WeatherNext natively carries the run's INIT time there -- one value for
# a whole run -- so a time lapse built on the raw collection collapses to
# a single frame. getForecastData restamps it on the way out, for every
# model, and these are the numbers that show it.
def frames(ic, label, show=4):
    d = ee.Dictionary({
        'n': ic.size(),
        'n_stamps': ic.aggregate_count_distinct('system:time_start'),
        'sts': ic.aggregate_array('system:time_start').sort(),
        'valid': ic.aggregate_array('valid_time').sort(),
        'leads': ic.aggregate_array('lead_hours').sort(),
    }).getInfo()
    utc = lambda ms: datetime.datetime.fromtimestamp(
        ms / 1000, datetime.timezone.utc).strftime('%Y-%m-%d %H:%M')
    stamps = [utc(t) for t in d['sts']]
    if len(stamps) <= 2 * show:
        line = ' | '.join(stamps)          # short enough to show whole
    else:
        line = (' | '.join(stamps[:show]) + '  ...  '
                + ' | '.join(stamps[-show:]))
    print(f"{label}: n={d['n']}, {d['n_stamps']} distinct system:time_start")
    print(f"    {line}")
    print(f"    system:time_start == valid_time: {d['sts'] == d['valid']}"
          f"   one stamp per image: {d['n_stamps'] == d['n']}"
          f"   leads {d['leads'][0]}..{d['leads'][-1]}")

gfs_fut  = wx.getForecastData(fut_start, fut_end, 'gfs')
gfs_past = wx.getForecastData(past_start, past_end, 'gfs')

frames(gfs_fut,  'gfs future ')   # one run, leads climbing
frames(gfs_past, 'gfs past   ')   # analyses only, one lead

# Hurricane Helene is September 2024, which only GFS reaches: ECMWF NRT
# begins 2024-11-12 and WeatherNext 3 begins in 2026. Those two return
# the empty-window sentinel for this range -- one fully masked image
# carrying lead_hours = -1, so downstream code cannot die on .first().
for m in ('euro', 'weathernext'):
    if m == 'weathernext' and not HAS_WN:
        continue
    ic = wx.getForecastData(past_start, past_end, m)
    d = ee.Dictionary({'n': ic.size(),
                       'lead': ic.aggregate_min('lead_hours')}).getInfo()
    tag = 'EMPTY (sentinel)' if d['lead'] == -1 else 'data'
    print(f"{m:12s} n={d['n']:3d} lead={d['lead']:4d}  {tag}"
          f"   <- archive does not go back to {past_start}")


gfs future n= 49  lead={'hi': 60, 'lo': 12}   <- one run
gfs past   n=  5  lead={'hi': 0, 'lo': 0}   <- analyses only
euro         n=  1 for 2024-09-26  (0 = archive does not go back this far)
weathernext  n=  1 for 2024-09-26  (0 = archive does not go back this far)


### A window that spans now

Ask for yesterday through three days out and you get both halves,
seamed at the most recent initialization: shortest-lead analyses up to
that moment, then that single run's forecast onward. The lead hours show
the join — they start at the analysis lead and climb through a real
forecast horizon.


In [3]:
span_start = (now - datetime.timedelta(days=2)).strftime('%Y-%m-%d')
span_end   = (now + datetime.timedelta(days=3)).strftime('%Y-%m-%d')

for m in (['gfs', 'euro'] + (['weathernext'] if HAS_WN else [])):
    frames(wx.getForecastData(span_start, span_end, m), f'{m:12s}')

print('')
print(f'asked {span_start} .. {span_end}  (now is {now:%m-%d %H:%M} UTC)')
print('The stamps run continuously across the seam: the analyses and the')
print('forecast run are different sources, but one ordered time axis.')


gfs          n=  71  09-08 00:00 -> 09-13 00:00  lead 0..60
euro         n=  31  09-08 00:00 -> 09-13 00:00  lead 0..60
weathernext  n= 119  09-08 01:00 -> 09-13 00:00  lead 1..60

asked 2026-09-08 .. 2026-09-13  (now is 09-10 20:26 UTC)


## 2. Wind: a speed raster plus animated particles

`Map.addWindLayer` adds **two** layers, the way windy.com does — a
smooth speed raster carrying the reading, with particle trails over it
showing the flow:

1. **`… speed`** — bicubic raster, and the layer a click reads. It
   carries both `speed` and `direction`.
2. **`… particles`** — an animated canvas. The wind components are
   decoded from Earth Engine PNG tiles (u in red, v in green, stretched
   ±40 m/s), so the particles follow the map **anywhere you pan**
   rather than being confined to a region fixed up front.

`viz` follows `addLayer`'s conventions. `bands` defaults to the image's
**first two bands in order** (dx, dy) — by position, because every
product names its components differently.

In [7]:
# One instant, not an average: the vector mean of a veering wind is not
# a wind anyone experiences.
Map.clearMap()
wind_img = ee.Image(gfs_past.sort('valid_time',False).first())

speed_dir, tiles = Map.addWindLayer(wind_img, {
    # ---- the raster half: what a click reads -------------------------
    'units': 'mi/hr',             # 'm/s' | 'km/hr' | 'mi/hr'.  default 'km/hr'
    'bands': ['u', 'v'],          # the dx/dy components; default = first two, in order
    'min': 0, 'max': 100,         # speed stretch. default max FOLLOWS THE UNIT
                                  #   (15 m/s | 54 km/hr | 34 mi/hr), so switching
                                  #   units cannot leave the raster one flat colour
    'palette': list(wx.WIND_PALETTE),   # default: windy.com's own ramp, 0-30 m/s
    'directionConvention': 'from',      # 'from' (default, meteorological: 270 is a
                                        #   westerly) | 'to' (where the air is going)

    # ---- colour ------------------------------------------------------
    'particleColor': '#fff',      # any CSS hex.  default '#fff'.  Dark basemaps
                                  #   want white; over pale terrain try '#222'
    'particleOpacity': 0.9,       # alpha at the HEAD. 0-1, default 0.9
                                  #   low 0.4 = ghostly | high 1.0 = hard-edged

    # ---- size (absolute pixel widths) --------------------------------
    'particleStrokeWeight': 1.1,  # base width in px. default 1.1
                                  #   min/max below default to 0.45x and 1.5x of it,
                                  #   so changing ONLY this rescales the whole taper
    'particleMinSize': 1.1 * 0.45,  # width at the TAIL, px. default strokeWeight*0.45
                                    #   low 0.2 = hairline tail | high = blunt ribbon
    'particleMaxSize': 1.1 * 1.5,   # width at the HEAD, px. default strokeWeight*1.5
                                    #   set equal to MinSize for a constant-width
                                    #   ribbon instead of a comet

    # ---- shape -------------------------------------------------------
    'particleTrailLength': 26,    # frames of history drawn. default 26
                                  #   THIS x the per-frame step IS the streak length
                                  #   low 8 = short dashes | high 60 = long ribbons
                                  #   (cost is linear: 60 draws ~2.3x the segments)
    'particleTaper': 2.1,         # exponent on the tail fade. default 2.1
                                  #   1.0 = linear wedge | >2 stretches the faint
                                  #   part out, which is what reads as a comet
    'particleHeadBoost': 1.6,     # alpha multiplier on the leading segment. default 1.6
                                  #   1.0 = no bright tip | 2+ = a hard spark
    'particleLineCap': 'round',   # 'round' (default, tapered tips) | 'butt' (blunt)

    # ---- speed -------------------------------------------------------
    'particleSpeedFactor': 380,   # seconds of advection per frame. default 380
                                  #   sets BOTH how fast the field moves and how long
                                  #   the streaks are.  low 150 = slow and short |
                                  #   high 800 = fast and long
                                  #   (screen length is held constant across zoom
                                  #    levels; see particleZoomRef)
    'particleMinSpeed': 3.0,      # apparent-speed FLOOR, m/s. default 3.0
                                  #   length is proportional to speed, so without a
                                  #   floor a light breeze draws a one-pixel dot and
                                  #   a calm map reads as a broken one.  0 disables
    'particleMaxSpeed': 45.0,     # apparent-speed CEILING, m/s. default 45.0
                                  #   without it a cyclone core smears clear across
                                  #   the screen.  lower to ~25 to tame hurricanes
    # NOTE both bounds change only how far a dot MOVES. Direction is untouched, and
    # the speed raster and the click query still report the true value -- so never
    # read a wind speed off a streak length.

    # ---- sampling ----------------------------------------------------
    'particleMaxTileZoom': 10,    # ceiling on the zoom of the u/v tiles. default 10
                                  #   tiles TRACK THE MAP so that geeViz's server-side
                                  #   resample('bicubic') is evaluated near display
                                  #   resolution -- the client takes the nearest pixel
                                  #   and does no interpolation of its own.
                                  #   low 6 = coarse and blocky when zoomed in |
                                  #   high 12 = finer than the 28 km forecast grid,
                                  #   i.e. pure upsampling at 4x the tiles per level.
                                  #   NOTE this does not bound how MANY tiles are
                                  #   fetched -- off-screen particles are culled, which
                                  #   holds a view to the ~35 tiles it can display.

    # ---- lifetime ----------------------------------------------------
    'particleMinAge': 22.5,       # shortest lifetime, frames. default maxAge * 0.25
    'particleMaxAge': 90,         # longest lifetime, frames. default 90
                                  #   each particle draws its OWN lifetime from this
                                  #   range, which is what puts short streaks
                                  #   alongside long ones. Set equal for a uniform
                                  #   comb.  Very low (<15) makes the field flicker.

    # ---- count -------------------------------------------------------
    'particleDensity': 1.75,      # particles per pixel of CANVAS WIDTH. default 1.75
                                  #   -> ~2975 on a 1700px canvas, ~3360 at 1920px.
                                  #   low 0.8 = sparse and cheap | high 4 = dense.
                                  #   This is the main cost knob.  Count does NOT
                                  #   vary with zoom: streak length is held constant
                                  #   on screen, so density should be too.
    # 'particleCount': 3000,      # overrides the density derivation outright.
                                  #   Omitted by default so the canvas is measured.
    'particleMinCount': 400,      # floor on the derived count. default 400
    'particleMaxCount': 20000,    # ceiling on it. default 20000
    'particleZoomRef': 7,         # the zoom streak length is CALIBRATED at. default 7
                                  #   every other zoom is scaled to match, so streaks
                                  #   stay the same SIZE ON SCREEN however far you
                                  #   zoom. Raise it to make streaks longer overall.

    # ---- field -------------------------------------------------------
    'particleFieldSpacing': 8,    # grid spacing, canvas px, of the wind field the
                                  #   client builds ONCE PER VIEW. default 8.
                                  #   The wind is resolved per cell -- projection,
                                  #   cos(lat), speed clamp, tile read -- and each
                                  #   particle then just reads the grid, so the
                                  #   per-frame cost is one array lookup.
                                  #   low 4 = finer flow, 4x the build cost |
                                  #   high 16 = cheaper, visibly coarser eddies
}, name='GFS 10 m wind')

# Units drive the stretch: a 0..15 scale read as km/h paints the map one
# flat colour, so max defaults per unit (15 m/s, 54 km/h, 34 mi/h).
print({u: wx.DEFAULT_MAX_SPEED[u] for u in wx.SPEED_UNITS})
Map.setCenter(-84,28.4,7)
Map.view(True)

Adding layer: GFS 10 m wind speed
Adding layer: GFS 10 m wind particles
{'m/s': 15.0, 'km/hr': 54.0, 'mi/hr': 34.0}
Starting webmap
Using eeCreds proxy at http://127.0.0.1:8889/ee-api (creds=ee-persistent, mode=detached via default)
geeView URL: http://127.0.0.1:8889/geeView/?v=1789072129642


### What a click reports

Direction is meteorological by default — the bearing the wind blows
FROM, so 270 is a westerly, which is what forecast products and barb
charts mean. Pass `'directionConvention': 'to'` for the way the air is
moving, which is what a spread model wants.

In [ ]:
denver = ee.Geometry.Point([-104.99, 39.74])
print(speed_dir.reduceRegion(ee.Reducer.first(), denver, 27830).getInfo())

## 3. The other models

Same call, different image.

In [ ]:
euro_fut = wx.getForecastData(fut_start, fut_end, 'euro')
Map.addWindLayer(ee.Image(euro_fut.sort('valid_time').first()),
                 {'units': 'km/hr', 'particleColor': '#ffe066'},
                 name='ECMWF 10 m wind', visible=False)

if HAS_WN:
    wn_fut = wx.getForecastData(fut_start, fut_end, 'weathernext')
    Map.addWindLayer(ee.Image(wn_fut.sort('valid_time').first()),
                     {'units': 'km/hr', 'particleColor': '#8ecae6'},
                     name='WeatherNext 3 wind', visible=False)

## 4. Variables across models — and the units trap

**GFS and ECMWF publish 2 m temperature in Celsius; WeatherNext
publishes Kelvin.** Measured at Denver for the same hour: 29.46 / 27.46
/ 297.40. Charting them raw puts one line 273 units off the others,
which reads as a model blow-up rather than a unit mismatch.
`getVariable` normalizes.

Where a model does not publish a variable it raises, rather than
returning an empty layer — ECMWF and WeatherNext publish dewpoint
instead of relative humidity, for instance.

In [ ]:
import json as _json
for name, entry in wx.VARIABLES.items():
    have = {m: (entry[m][0] if entry.get(m) else None)
            for m in ('euro', 'gfs', 'weathernext')}
    print(f'{name:24s} {_json.dumps(have)}')

try:
    wx.getVariable(None, 'relative_humidity_2m', 'euro')
except ValueError as e:
    print(f'\neuro relative humidity -> {e}')

In [ ]:
RAW = {m: ee.ImageCollection(wx.MODELS[m]['collection'])
       for m in ('euro', 'gfs', 'weathernext')}

# wx ships windy.com's own ramps, so a geeViz map and a windy map of
# the same hour read the same way.
TEMP_VIZ = {'min': -5, 'max': 40,
            'palette': list(wx.TEMPERATURE_PALETTE),
            'yLabel': 'Temperature (C)'}

for model in ('gfs', 'euro'):
    t = wx.getVariable(RAW[model].filterDate(fut_start, fut_end),
                       'temperature_2m', model)
    Map.addLayer(t.mosaic().clip(study_area), TEMP_VIZ,
                 f'{model.upper()}: Temperature (2 m)', model == 'gfs')

rh = wx.getVariable(RAW['gfs'].filterDate(fut_start, fut_end),
                    'relative_humidity_2m', 'gfs')
Map.addLayer(rh.mosaic().clip(study_area),
             {'min': 0, 'max': 100, 'palette': 'd73027,ffffbf,4575b4',
              'yLabel': 'Relative humidity (%)'},
             'GFS: Relative humidity (2 m)', False)

precip = wx.getVariable(RAW['gfs'].filterDate(fut_start, fut_end),
                        'precipitation', 'gfs')
Map.addLayer(precip.mosaic().clip(study_area),
             {'min': 0, 'max': 0.0005,
              'palette': list(wx.PRECIP_PALETTE),
              'yLabel': 'Precipitation rate (kg/m^2/s)'},
             'GFS: Precipitation rate', False)

> **GFS bands are not stable across the collection.** Some images
> carry `total_precipitation_surface`; others carry `precipitation_rate`
> alongside `gust`, `haines_index` and `ventilation_rate`. If a select
> fails, inspect that image rather than assuming the band is gone.

## 5. WeatherNext ensemble percentiles

The 64 members are published **pre-aggregated** as `_mean`, `_p10`,
`_p25`, `_p50`, `_p75`, `_p90`. So forecast spread is the difference of
two of these, not a reduction over members — which is both cheaper and
the only option, since the members themselves are not in the collection.

In [ ]:
if HAS_WN:
    wn = RAW['weathernext'].filterDate(fut_start, fut_end)
    t10 = wx.getVariable(wn, 'temperature_2m', 'weathernext', stat='p10')
    t90 = wx.getVariable(wn, 'temperature_2m', 'weathernext', stat='p90')
    spread = t90.mosaic().subtract(t10.mosaic()).rename('spread')
    Map.addLayer(spread.clip(study_area),
                 {'min': 0, 'max': 10,
                  'palette': '000004,420a68,932667,dd513a,fca50a,fcffa4',
                  'yLabel': 'p90 - p10 (C)'},
                 'WeatherNext 3: temperature spread', False)
    print('spread = p90 - p10; widens with lead time as confidence drops')

## 6. View the map

Toggle layers in the panel and click anywhere for a query. Watch the
particles: they re-seed to the view on every pan, and because the
components come from tiles rather than a fixed lattice, they keep
flowing wherever you go.

In [ ]:
Map.turnOnInspector()
Map.view()